# Mundiales 2018, 2022 y 2026
## Preparación de datos y entrada al análisis supervisado

Trabajaremos exclusivamente con la fase de grupos. Los archivos contienen errores deliberados. No uses la base del profesor.

## Objetivos

- Perfilar datos.
- Estandarizar esquemas.
- Limpiar fechas, equipos, goles y marcadores.
- Eliminar duplicados con criterio.
- Comparar torneos mediante tasas.
- Construir variables conocidas antes de cada partido.
- Entrenar un primer clasificador y detectar fuga de información.

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

DATA = Path('../datos')
pd.set_option('display.max_columns', 50)

## Cargar los tres archivos

In [ ]:
d18 = pd.read_csv(DATA / 'mundial_2018_sucio.csv', dtype=str)
d22 = pd.read_csv(DATA / 'mundial_2022_sucio.csv', dtype=str)
d26 = pd.read_csv(DATA / 'mundial_2026_sucio.csv', dtype=str)

for nombre, df in [('2018', d18), ('2022', d22), ('2026', d26)]:
    print(nombre, df.shape)
    display(df.head(3))


## Perfilado

Para cada archivo revisa:

- columnas;
- tipos;
- valores nulos;
- duplicados;
- valores únicos de grupos, fases y equipos;
- goles que no puedan convertirse directamente a número.

In [ ]:
def perfil(df, nombre):
    print(f"--- {nombre} ---")
    print("filas x columnas:", df.shape)
    print("columnas:", list(df.columns))
    print("nulos por columna:\n", df.isna().sum())
    print("duplicados exactos:", df.duplicated().sum())
    for col in df.columns:
        if df[col].nunique() <= 15:
            print(f"  valores únicos de {col}:", sorted(df[col].dropna().unique().tolist()))
    print()

perfil(d18, '2018')
perfil(d22, '2022')
perfil(d26, '2026')


## Unificar nombres de columnas

In [ ]:
rename_maps = {
    2018: {
        'ID Partido': 'partido_id', 'Año': 'mundial', 'Fase': 'fase',
        'Grupo': 'grupo', 'Jornada': 'jornada', 'Fecha': 'fecha',
        'Equipo Local': 'equipo_local', 'Equipo Visitante': 'equipo_visitante',
        'Goles Local': 'goles_local', 'Goles Visitante': 'goles_visitante',
        'Marcador': 'marcador', 'Anfitrión Local': 'local_es_anfitrion',
        'Fuente': 'fuente',
    },
    2022: {
        'match_id': 'partido_id', 'WorldCup': 'mundial', 'stage': 'fase',
        'group_name': 'grupo', 'match_day': 'jornada', 'date': 'fecha',
        'local': 'equipo_local', 'visitor': 'equipo_visitante',
        'home_score': 'goles_local', 'away_score': 'goles_visitante',
        'score_text': 'marcador', 'home_host': 'local_es_anfitrion',
        'source_url': 'fuente',
    },
    2026: {
        'match': 'partido_id', 'wc': 'mundial', 'round': 'fase',
        'grp': 'grupo', 'md': 'jornada', 'played_on': 'fecha',
        'home': 'equipo_local', 'away': 'equipo_visitante',
        'HG': 'goles_local', 'AG': 'goles_visitante',
        'result_raw': 'marcador', 'host_h': 'local_es_anfitrion',
        'host_a': 'visitante_es_anfitrion', 'source': 'fuente',
    },
}

# Esquema canónico mínimo:
columnas_base = [
    'partido_id', 'mundial', 'fase', 'grupo', 'jornada', 'fecha',
    'equipo_local', 'equipo_visitante', 'goles_local',
    'goles_visitante', 'marcador', 'local_es_anfitrion',
    'visitante_es_anfitrion', 'fuente'
]


## Normalizar equipos

No conviene borrar acentos del nombre que se mostrará. Crea una clave auxiliar sin acentos, minúscula y sin signos para buscar en el catálogo.

In [ ]:
catalogo = pd.read_csv(DATA / 'catalogo_equipos.csv')

def clave_texto(valor):
    # Minúsculas, sin acentos, sin signos -> clave para cruzar con el catálogo.
    if pd.isna(valor):
        return ""
    v = str(valor).strip()
    v = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii")
    v = v.lower()
    v = re.sub(r"[^a-z0-9]+", " ", v)
    v = re.sub(r"\s+", " ", v).strip()
    return v

# diccionario clave_normalizada -> nombre_canonico
mapa_equipos = {
    clave_texto(row.variante): row.nombre_canonico
    for row in catalogo.itertuples()
}

def normalizar_equipo(valor):
    k = clave_texto(valor)
    return mapa_equipos.get(k, str(valor).strip() if pd.notna(valor) else valor)


## Fechas, grupos, booleanos y marcadores

In [ ]:
rangos = {
    2018: ('2018-06-14', '2018-06-28'),
    2022: ('2022-11-20', '2022-12-02'),
    2026: ('2026-06-11', '2026-06-27'),
}

MESES = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
    "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}

def convertir_fecha(valor, mundial):
    # 1. reconoce seriales de Excel; 2. prueba varios formatos;
    # 3. elige la fecha que cae dentro del rango del torneo.
    lo, hi = pd.Timestamp(rangos[mundial][0]), pd.Timestamp(rangos[mundial][1])
    if pd.isna(valor):
        return pd.NaT
    v = str(valor).strip()

    if re.fullmatch(r"\d{4,6}", v):
        try:
            fecha = pd.Timestamp("1899-12-30") + pd.Timedelta(days=int(v))
            if lo <= fecha <= hi:
                return fecha
        except Exception:
            pass

    m = re.fullmatch(r"([A-Za-z]{3})[a-z]*\.?\s+(\d{1,2}),?\s+(\d{4})", v)
    if m:
        mes = MESES.get(m.group(1).lower()[:3])
        if mes:
            try:
                return pd.Timestamp(year=int(m.group(3)), month=mes, day=int(m.group(2)))
            except Exception:
                pass

    candidatos = []
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%d-%m-%y", "%m-%d-%y"):
        try:
            candidatos.append(pd.to_datetime(v, format=fmt))
        except Exception:
            continue
    for c in candidatos:
        if lo <= c <= hi:
            return c
    return candidatos[0] if candidatos else pd.NaT

def extraer_numero(valor):
    # Devuelve el primer entero no negativo del texto; 's/d', '' -> NaN.
    if pd.isna(valor):
        return np.nan
    v = str(valor).strip().lower()
    if v in {"", "s/d", "nan", "n/a"}:
        return np.nan
    m = re.search(r"-?\d+", v)
    if not m:
        return np.nan
    n = int(m.group())
    return np.nan if n < 0 else n

def separar_marcador(valor):
    # Acepta -, –, —, :, x, X
    if pd.isna(valor):
        return (np.nan, np.nan)
    v = str(valor).strip()
    m = re.match(r"^\s*(\d+)\s*[-–—:xX]\s*(\d+)\s*$", v)
    if not m:
        return (np.nan, np.nan)
    return (int(m.group(1)), int(m.group(2)))

def a_booleano(valor):
    if pd.isna(valor):
        return np.nan
    v = str(valor).strip().lower()
    if v in {"si", "sí", "true", "1", "yes", "x"}:
        return True
    if v in {"no", "false", "0", ""}:
        return False
    return np.nan

def normalizar_grupo(valor):
    if pd.isna(valor):
        return np.nan
    v = str(valor).strip().lower()
    v = v.replace("grupo", "").replace("group", "").replace("group-", "")
    v = re.sub(r"[^a-z]", "", v)
    return v.upper()


## Función de limpieza reproducible

In [ ]:
def limpiar_mundial(df, mundial):
    df = df.rename(columns=rename_maps[mundial])
    df = df[[c for c in columnas_base if c in df.columns]].copy()

    # Solo 2026 trae columna propia para el anfitrión visitante
    # (2018 y 2022 tuvieron un único país anfitrión).
    if 'visitante_es_anfitrion' not in df.columns:
        df['visitante_es_anfitrion'] = False

    df['mundial'] = mundial
    df['fase'] = 'Fase de grupos'
    df['grupo'] = df['grupo'].apply(normalizar_grupo)
    df['jornada'] = df['jornada'].apply(extraer_numero).astype('Int64')

    df['equipo_local'] = df['equipo_local'].apply(normalizar_equipo)
    df['equipo_visitante'] = df['equipo_visitante'].apply(normalizar_equipo)

    df['fecha'] = df['fecha'].apply(lambda v: convertir_fecha(v, mundial))

    df['local_es_anfitrion'] = df['local_es_anfitrion'].apply(a_booleano).fillna(False)
    df['visitante_es_anfitrion'] = df['visitante_es_anfitrion'].apply(a_booleano).fillna(False)

    df['partido_id'] = df['partido_id'].str.strip()
    df['fuente'] = df['fuente'].str.strip()

    # Se prioriza el marcador para reparar los goles; si no puede
    # interpretarse, se usan las columnas de goles sueltas.
    marcador_local, marcador_visita = zip(*df['marcador'].apply(separar_marcador))
    goles_local_col = df['goles_local'].apply(extraer_numero)
    goles_visita_col = df['goles_visitante'].apply(extraer_numero)

    df['goles_local'] = pd.Series(marcador_local, index=df.index).combine_first(goles_local_col)
    df['goles_visitante'] = pd.Series(marcador_visita, index=df.index).combine_first(goles_visita_col)

    df['marcador'] = df.apply(
        lambda r: f"{int(r.goles_local)}-{int(r.goles_visitante)}"
        if pd.notna(r.goles_local) and pd.notna(r.goles_visitante) else np.nan,
        axis=1,
    )

    # Inferir el grupo faltante viendo en qué grupo aparece ese equipo
    # en otras filas del mismo torneo.
    conocido = df.dropna(subset=['grupo'])
    equipo_a_grupo = {}
    for r in conocido.itertuples():
        equipo_a_grupo.setdefault(r.equipo_local, r.grupo)
        equipo_a_grupo.setdefault(r.equipo_visitante, r.grupo)
    mask_sin_grupo = df['grupo'].isna() | (df['grupo'] == '')
    df.loc[mask_sin_grupo, 'grupo'] = df.loc[mask_sin_grupo].apply(
        lambda r: equipo_a_grupo.get(r.equipo_local) or equipo_a_grupo.get(r.equipo_visitante),
        axis=1,
    )

    df = df.drop_duplicates(subset=['partido_id'], keep='first')

    df['resultado_local'] = np.select(
        [df.goles_local > df.goles_visitante, df.goles_local < df.goles_visitante],
        ['Gana', 'Pierde'], default='Empata',
    )
    df['goles_totales'] = df['goles_local'] + df['goles_visitante']
    df['diferencia_goles'] = df['goles_local'] - df['goles_visitante']

    return df.reset_index(drop=True)

limpio18 = limpiar_mundial(d18, 2018)
limpio22 = limpiar_mundial(d22, 2022)
limpio26 = limpiar_mundial(d26, 2026)

print(limpio18.shape, limpio22.shape, limpio26.shape)
partidos = pd.concat([limpio18, limpio22, limpio26], ignore_index=True)
partidos.head()


## Validaciones obligatorias

La limpieza no termina cuando el código deja de producir errores. Debes comprobar invariantes.

In [ ]:
conteo = partidos.groupby('mundial').size().to_dict()
assert conteo == {2018: 48, 2022: 48, 2026: 72}, f"Conteo inesperado: {conteo}"

assert not partidos['partido_id'].duplicated().any(), "Hay partido_id duplicados"

assert (partidos['goles_local'] >= 0).all(), "Hay goles_local negativos"
assert (partidos['goles_visitante'] >= 0).all(), "Hay goles_visitante negativos"

cols_criticas = ['equipo_local', 'equipo_visitante', 'goles_local', 'goles_visitante', 'grupo']
nulos = partidos[cols_criticas].isna().sum()
assert nulos.sum() == 0, f"Hay nulos:\n{nulos[nulos > 0]}"

marcador_esperado = (
    partidos['goles_local'].astype('Int64').astype(str) + '-' +
    partidos['goles_visitante'].astype('Int64').astype(str)
)
assert (partidos['marcador'] == marcador_esperado).all(), "El marcador no coincide con los goles"

print("Todas las validaciones pasaron correctamente ✔")
print(conteo)


## Comparación de los Mundiales

In [ ]:
comparacion = partidos.groupby('mundial').agg(
    partidos=('partido_id', 'count'),
    goles=('goles_totales', 'sum'),
    empates=('resultado_local', lambda s: (s == 'Empata').sum()),
).reset_index()

comparacion['goles_por_partido'] = comparacion['goles'] / comparacion['partidos']
comparacion['porcentaje_empates'] = 100 * comparacion['empates'] / comparacion['partidos']

# Victorias del anfitrión: gana si (es local y gana) o (es visitante y pierde el local)
def gana_anfitrion(row):
    if row.local_es_anfitrion and row.resultado_local == 'Gana':
        return True
    if row.visitante_es_anfitrion and row.resultado_local == 'Pierde':
        return True
    return False

partidos['anfitrion_gana'] = partidos.apply(gana_anfitrion, axis=1)
partidos['hay_anfitrion'] = partidos['local_es_anfitrion'] | partidos['visitante_es_anfitrion']

anfitrion = partidos[partidos['hay_anfitrion']].groupby('mundial').agg(
    partidos_con_anfitrion=('partido_id', 'count'),
    victorias_anfitrion=('anfitrion_gana', 'sum'),
).reset_index()
anfitrion['porcentaje_victorias_anfitrion'] = (
    100 * anfitrion['victorias_anfitrion'] / anfitrion['partidos_con_anfitrion']
)

comparacion = comparacion.merge(
    anfitrion[['mundial', 'porcentaje_victorias_anfitrion']], on='mundial', how='left'
)
comparacion['prop_mas_2_5_goles'] = (
    partidos.groupby('mundial')['goles_totales'].apply(lambda s: (s > 2.5).mean()).values * 100
)

# Equipo con mejor diferencia de goles por torneo (requiere tabla_equipos, ver siguiente celda)
display(comparacion.round(2))

# Gráficos: no comparar solo goles totales, 2026 tiene más partidos
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(comparacion['mundial'].astype(str), comparacion['goles'], color='#4C72B0')
axes[0].set_title('Goles totales por torneo')
axes[1].bar(comparacion['mundial'].astype(str), comparacion['goles_por_partido'], color='#55A868')
axes[1].set_title('Goles por partido (tasa)')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
partidos['resultado_local'].value_counts().unstack if False else None
tabla_resultados = partidos.groupby(['mundial', 'resultado_local']).size().unstack(fill_value=0)
tabla_resultados = tabla_resultados[['Gana', 'Empata', 'Pierde']]
tabla_resultados_pct = tabla_resultados.div(tabla_resultados.sum(axis=1), axis=0) * 100
tabla_resultados_pct.plot(kind='bar', stacked=True, ax=ax, color=['#4C72B0', '#DD8452', '#C44E52'])
ax.set_title('Distribución de resultados por mundial (%)')
ax.set_ylabel('%')
plt.tight_layout()
plt.show()


## Tabla por equipo

In [ ]:
locales = partidos.rename(columns={
    'equipo_local': 'equipo', 'equipo_visitante': 'rival',
    'goles_local': 'gf', 'goles_visitante': 'gc',
})[['mundial', 'equipo', 'rival', 'gf', 'gc', 'resultado_local']].copy()
locales['resultado'] = locales['resultado_local'].map({'Gana': 'PG', 'Empata': 'PE', 'Pierde': 'PP'})

visitantes = partidos.rename(columns={
    'equipo_visitante': 'equipo', 'equipo_local': 'rival',
    'goles_visitante': 'gf', 'goles_local': 'gc',
})[['mundial', 'equipo', 'rival', 'gf', 'gc', 'resultado_local']].copy()
# Desde la óptica del visitante el resultado del local se invierte
visitantes['resultado'] = visitantes['resultado_local'].map({'Gana': 'PP', 'Empata': 'PE', 'Pierde': 'PG'})

apariciones = pd.concat([locales, visitantes], ignore_index=True).drop(columns=['resultado_local'])

tabla_equipos = apariciones.groupby(['mundial', 'equipo']).agg(
    PJ=('equipo', 'count'),
    PG=('resultado', lambda s: (s == 'PG').sum()),
    PE=('resultado', lambda s: (s == 'PE').sum()),
    PP=('resultado', lambda s: (s == 'PP').sum()),
    GF=('gf', 'sum'),
    GC=('gc', 'sum'),
).reset_index()

tabla_equipos['DG'] = tabla_equipos['GF'] - tabla_equipos['GC']
tabla_equipos['PTS'] = tabla_equipos['PG'] * 3 + tabla_equipos['PE']
tabla_equipos['PTS_por_partido'] = tabla_equipos['PTS'] / tabla_equipos['PJ']

tabla_equipos = tabla_equipos.sort_values(['mundial', 'PTS', 'DG'], ascending=[True, False, False])

print("Equipo con mejor diferencia de goles por torneo:")
display(tabla_equipos.loc[tabla_equipos.groupby('mundial')['DG'].idxmax()])

print()
print("Tabla completa (primeros 3 de cada grupo de mundial, por PTS):")
display(tabla_equipos.groupby('mundial').head(3))


## Variables previas al partido

Para predecir no podemos utilizar datos ocurridos después del inicio. Crearemos promedios acumulados antes de cada encuentro.

In [ ]:
def construir_variables_previas(partidos):
    partidos = partidos.sort_values(['mundial', 'fecha', 'jornada', 'partido_id']).reset_index(drop=True)

    # Estado acumulado por (mundial, equipo): PJ, puntos, GF, GC
    estado = {}
    filas = []

    for r in partidos.itertuples():
        clave_local = (r.mundial, r.equipo_local)
        clave_visita = (r.mundial, r.equipo_visitante)

        e_local = estado.get(clave_local, {'PJ': 0, 'PTS': 0, 'GF': 0, 'GC': 0})
        e_visita = estado.get(clave_visita, {'PJ': 0, 'PTS': 0, 'GF': 0, 'GC': 0})

        def promedios(e):
            if e['PJ'] == 0:
                return 0.0, 0.0, 0.0  # sin historial -> 0, no NaN
            return e['PTS'] / e['PJ'], (e['GF'] - e['GC']) / e['PJ'], e['GF'] / e['PJ']

        local_pts_prom_pre, local_gd_prom_pre, local_gf_prom_pre = promedios(e_local)
        visita_pts_prom_pre, visita_gd_prom_pre, visita_gf_prom_pre = promedios(e_visita)

        filas.append({
            'partido_id': r.partido_id, 'mundial': r.mundial, 'jornada': r.jornada,
            'equipo_local': r.equipo_local, 'equipo_visitante': r.equipo_visitante,
            'local_pts_prom_pre': local_pts_prom_pre, 'visita_pts_prom_pre': visita_pts_prom_pre,
            'local_gd_prom_pre': local_gd_prom_pre, 'visita_gd_prom_pre': visita_gd_prom_pre,
            'local_gf_prom_pre': local_gf_prom_pre, 'visita_gf_prom_pre': visita_gf_prom_pre,
            'local_es_anfitrion': r.local_es_anfitrion, 'visitante_es_anfitrion': r.visitante_es_anfitrion,
            'goles_local': r.goles_local, 'goles_visitante': r.goles_visitante,
            'diferencia_goles': r.diferencia_goles, 'resultado_local': r.resultado_local,
        })

        # El estado se actualiza DESPUÉS de registrar los promedios "pre".
        # Esto es lo que evita la fuga de información.
        pts_local = 3 if r.resultado_local == 'Gana' else (1 if r.resultado_local == 'Empata' else 0)
        pts_visita = 3 if r.resultado_local == 'Pierde' else (1 if r.resultado_local == 'Empata' else 0)

        e_local['PJ'] += 1; e_local['PTS'] += pts_local
        e_local['GF'] += r.goles_local; e_local['GC'] += r.goles_visitante
        e_visita['PJ'] += 1; e_visita['PTS'] += pts_visita
        e_visita['GF'] += r.goles_visitante; e_visita['GC'] += r.goles_local

        estado[clave_local] = e_local
        estado[clave_visita] = e_visita

    return pd.DataFrame(filas)

features_df = construir_variables_previas(partidos)
display(features_df.head())


## Entrenamiento y prueba

In [ ]:
features = [
    'jornada',
    'local_pts_prom_pre', 'visita_pts_prom_pre',
    'local_gd_prom_pre', 'visita_gd_prom_pre',
    'local_gf_prom_pre', 'visita_gf_prom_pre',
    'local_es_anfitrion', 'visitante_es_anfitrion'
]
objetivo = 'resultado_local'

train_df = features_df[features_df['mundial'].isin([2018, 2022])].copy()
test_df = features_df[features_df['mundial'] == 2026].copy()

X_train, y_train = train_df[features].astype(float), train_df[objetivo]
X_test, y_test = test_df[features].astype(float), test_df[objetivo]

# Línea base: predecir siempre la clase más frecuente del entrenamiento
clase_mas_frecuente = y_train.value_counts().idxmax()
baseline_acc = accuracy_score(y_test, [clase_mas_frecuente] * len(y_test))
print("Línea base (predice siempre '" + clase_mas_frecuente + "'):", round(baseline_acc, 3))

modelo = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Árbol de decisión (sin fuga):", round(acc, 3))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=['Gana', 'Empata', 'Pierde'], ax=ax)
ax.set_title("Matriz de confusión - prueba 2026")
plt.show()

fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(modelo, feature_names=features, class_names=modelo.classes_, filled=True, fontsize=8, ax=ax)
plt.show()


## Experimento de fuga de información

Agrega temporalmente `goles_local`, `goles_visitante` y `diferencia_goles` como variables. Si la precisión sube de forma extrema, explica por qué el modelo no está prediciendo realmente.

In [ ]:
features_fuga = features + ['goles_local', 'goles_visitante', 'diferencia_goles']

X_train_f = train_df[features_fuga].astype(float)
X_test_f = test_df[features_fuga].astype(float)

modelo_fuga = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
modelo_fuga.fit(X_train_f, y_train)
y_pred_fuga = modelo_fuga.predict(X_test_f)
acc_fuga = accuracy_score(y_test, y_pred_fuga)

print(f"Accuracy SIN fuga: {acc:.3f}")
print(f"Accuracy CON fuga: {acc_fuga:.3f}")
print()
print("'diferencia_goles' determina casi por completo 'resultado_local'")
print("(>0 -> Gana, =0 -> Empata, <0 -> Pierde). El modelo con fuga no está")
print("aprendiendo a predecir el partido: está leyendo el resultado desde una")
print("variable que solo existe DESPUÉS de jugarse. Una precisión cercana al")
print("100% con variables que no deberían conocerse antes del partido es la")
print("señal de alarma clásica de data leakage.")


## Reflexión final

Responde:

- ¿Qué problema de calidad fue el más difícil?
- ¿Qué decisión de limpieza podría cambiar los resultados?
- ¿Por qué 2026 debe compararse mediante tasas?
- ¿El árbol supera la línea base?
- ¿Qué variables reales agregarías para mejorar una predicción?
- ¿Por qué un resultado de 100 % puede ser una señal de alarma?